
# ML: predict credit owners with high default probability

Once all data is loaded and secured (the **data unification** part), we can proceed to exploring, understanding, and using the data to create actionable insights - **data decisioning**.


As outlined in the [introductory notebook]($../00-Credit-Decisioning), we will build machine learning (ML) models for driving three business outcomes:
1. Identify currently underbanked customers with high credit worthiness so we can offer them credit instruments,
2. Predict current credit owners with high probability of defaulting along with the loss-given default, and
3. Offer instantaneous micro-loans (Buy Now, Pay Later) when a customer does not have the required credit limit or account balance to complete a transaction.

Here is the flow we'll implement: 

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/credit_decisioning/fsi-credit-decisioning-ml-0.png" width="1200px">


<!-- Collect usage data (view). Remove it to disable collection. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F03-Data-Science-ML%2F03.1-Feature-Engineering-credit-decisioning&demo_name=lakehouse-fsi-credit&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-credit%2F03-Data-Science-ML%2F03.1-Feature-Engineering-credit-decisioning&version=1">


## The need for Enhanced Collaboration

Feature Engineering is an iterative process - we need to quickly generate new features, test the model, and go back to feature selection and more feature engineering - many many times. The Databricks Lakehouse enables data teams to collaborate extremely effectively through the following Databricks Notebook features:
1. Sharing and collaborating in the same Notebook by any team member (with different access modes),
2. Ability to use python, SQL, and R simultaneously in the same Notebook on the same data,
3. Native integration with a Git repository (including AWS Code Commit, Azure DevOps, GitLabs, Github, and others), making the Notebooks tools for CI/CD,
4. Variables explorer,
5. Automatic Data Profiling (in the cell below), and
6. GUI-based dashboards (in the cell below) that can also be added to any Databricks SQL Dashboard.

These features enable teams within FSI organizations to become extremely fast and efficient in building the best ML model at reduced time, thereby making the most out of market opportunities such as the raising interest rates.

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0 databricks-feature-store==0.17.0
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/569.1 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.1/569.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/27.4 MB ? eta -:--:--
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/27.4 MB 212.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━ 14.5/27.4 MB 225.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 22.5/27.4 MB 224.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 27.4/27.4 MB 222.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/5.9 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 5.8/5.9 MB 249.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 146.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/147.8 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.2 MB/s eta 0:00:00
  

In [0]:
%run ../_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_credit_decisioning`



## Data exploration & Features creation

<img src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/credit_decisioning/fsi-credit-decisioning-ml-1.png" style="float: right" width="800px">

<br/><br/>
The first step as Data Scientist is to explore our data and understand it to create Features.

<br/>

This where we use our existing tables and transform the data to be ready for our ML models. These features will later be stored in Databricks Feature Store (see below) and used to train the aforementioned ML models.

<br/>

Let's start with some data exploration. Databricks comes with built-in Data Profiling to help you bootstrap that.

In [0]:
%sql
SELECT * FROM customer_gold WHERE tenure_months BETWEEN 10 AND 150

id,first_name,last_name,email,gender,address,city,postal_code,home_phone,work_phone,mobile_phone,cust_type,join_date,status,education,marital_status,months_current_address,months_employment,is_resident,passport_expiry,visa_expiry,document_id,operation,cust_id,tenure_months,product_cnt,tot_rel_bal,claim_cnt,revenue_tot,revenue_12m,income_monthly,income_annual,tot_assets,debt_income_rt_w_mortgage,debt_income_rt_wo_mortgage,total_cards,total_loans,total_mortgages,card_balance_amount,loan_balance_amount,loan_limit_amount,card_limit_amount,overdraft_balance_amount,overdraft_number,total_deposits_number,total_deposits_amount,total_equity_amount,total_UT,customer_revenue,number_customer_enquries,customer_score,total_payment_made,relationship_manager,total_credit_outstanding,birth_year,avg_balance,num_accs,balance_usd,available_balance_usd
148,Francisca,Friday,ffriday43@google.ca,Male,6 David Way,Jammā‘īn,null,395-847-6653,684-686-5134,188-671-0737,2,2022-12-11,0,4,1,15,1,1,2029-02-20,2058-11-25,SY8430215,INSERT,148,67,3,1229.33,6,709.64,0.88,2361,28332,28332,0.36,0.44,0,0,4,7.5261403133E8,9.5740123006E8,9.4253349997E8,3.0083656132E8,2.9202690498E8,10,9,231697.88,1979774.37,9048360.0,8079228.0,5,null,null,null,null,2003,1002.01,4,581.33,2247.8
463,Keelby,Gayton,kgaytoncu@discuz.net,Male,4 Oriole Crossing,Zipárion,null,612-670-9738,null,580-522-8883,4,2004-08-06,1,4,1,17,1,0,2049-11-25,2096-12-05,MV3516847,INSERT,463,28,4,10159.61,4,1202.23,3.58,8119,97428,0,0.61,0.37,10,3,6,9.2534864944E8,3.1573068476E8,8.9078431784E8,9.4632208176E8,2.1270324083E8,4,6,8457003.0,8405245.0,8589421.0,7210788.0,4,null,null,null,null,1971,1181.865,2,2154.23,838.17
471,Emelita,Jikovsky,ejikovskyd2@umn.edu,Female,481 Shasta Point,Har-Us,null,338-446-2043,108-416-0663,156-236-8738,4,2010-07-10,0,0,0,5,1,0,2028-01-16,2057-07-10,KG6184964,INSERT,471,52,2,52.0,9,1467.79,2.35,8839,106068,1060,0.09,0.3,2,0,4,1.7354365081E8,5.544722093E7,9.505417703E7,2.2011149356E8,4.4269459686E8,8,9,1606463.0,5484075.09,3431393.0,7249848.02,6,65,4726740.74,12,2202678.51,1989,5238.02,1,5238.02,1924.28
496,Anjanette,Longstaff,alongstaffdr@moonfruit.com,Male,730 Maple Wood Way,Qelëz,null,null,null,640-630-4537,3,2002-01-06,1,0,0,1,18,0,2081-06-09,2064-10-04,YV1318504,INSERT,496,108,5,5081.31,0,49.16,0.04,5780,69360,69360,0.26,0.56,7,7,4,8.4165254118E8,4.5746814468E8,3.3987193586E8,3.7057081819E8,8.6094333863E8,5,4,9366416.0,7425938.0,6857042.78,4044357.0,0,28,6427137.0,0,4249166.76,1964,1424.46,1,1424.46,919.08
833,Byran,Francois,bfrancoisn4@icq.com,Female,01 Donald Terrace,Barreiras,47800-000,null,null,797-307-5774,4,2020-08-16,1,4,0,17,33,1,2029-12-28,2046-09-14,DM6348720,INSERT,833,147,1,4171.9,0,946.31,0.54,7962,95544,955,0.5,0.49,4,5,10,6.3378530634E8,8.9855442163E8,5.0970471492E8,6.6197900006E8,6.5867073154E8,10,8,4417594.81,1782286.12,3998838.0,7631038.0,6,null,null,null,null,1960,415.0,1,415.0,895.37
1088,Helyn,Beaven,hbeavenu7@bandcamp.com,Male,226 Becker Way,Gungguh,null,381-307-8797,null,856-786-9293,2,2022-04-30,1,2,2,53,49,0,2067-10-30,2022-11-20,null,INSERT,1088,52,4,11780.41,0,199.02,0.32,6254,75048,0,0.54,0.63,9,0,6,5.2609513805E8,1.436279725E8,9.8798453661E8,8.5574088141E8,9.3016475685E8,8,3,2275684.32,1252601.0,498446.29,7084018.0,10,null,null,null,null,1960,1332.28,1,1332.28,1262.51
1238,Gardie,Purser,gpurseryd@sphinn.com,Male,455 Raven Junction,El Ángel,null,null,769-630-3737,690-925-6781,2,2021-01-01,1,4,0,49,57,1,2051-12-09,2072-08-16,LR5944525,INSERT,1238,68,1,198.0,9,1363.77,1.67,3883,46596,0,0.8,0.71,6,5,1,8.1057971122E8,4.162891736E7,5.0088671848E8,4.42078943E7,1.1610549949E8,2,7,4674719.93,6838865.0,3268432.51,4055899.25,9,null,null,null,null,2001,261.22,1,261.22,250.14
1342,Doralynne,Pieterick,dpieterick119@fotki.com,Female,61722 Eastlawn Terrace,Jiudian,null,523-261-6903,834-243-8134,131-933-7955,3,2002-06-04,0,0,2,73,24,0,2024-05-12,2037-08-18,CE7689497,INSERT,1342,88,3,14296.34,7,135.68,0.13,9635,115620,0,0.22,0.09,4,9,7,5.8052861869E8,2.7299443199E8,9

In [0]:
data = spark.table("customer_gold") \
              .where("tenure_months BETWEEN 10 AND 150") \
              .groupBy("tenure_months", "education").sum("income_monthly") \
              .orderBy('education').toPandas()

px.bar(data, x="tenure_months", y="sum(income_monthly)", color="education", title="Wide-Form Input")


# Building our Features for Credit Default risks

To build our model predicting credit default risks, we'll need a buch of features. To improve our governance and centralize our data for multiple ML project, we can save our ML features using a Feature Store.

In [0]:
customer_gold_features = (spark.table("customer_gold")
                               .withColumn('age', int(date.today().year) - col('birth_year'))
                               .select('cust_id', 'education', 'marital_status', 'months_current_address', 'months_employment', 'is_resident',
                                       'tenure_months', 'product_cnt', 'tot_rel_bal', 'revenue_tot', 'revenue_12m', 'income_annual', 'tot_assets', 
                                       'overdraft_balance_amount', 'overdraft_number', 'total_deposits_number', 'total_deposits_amount', 'total_equity_amount', 
                                       'total_UT', 'customer_revenue', 'age', 'avg_balance', 'num_accs', 'balance_usd', 'available_balance_usd')).dropDuplicates(['cust_id'])
display(customer_gold_features)

cust_id,education,marital_status,months_current_address,months_employment,is_resident,tenure_months,product_cnt,tot_rel_bal,revenue_tot,revenue_12m,income_annual,tot_assets,overdraft_balance_amount,overdraft_number,total_deposits_number,total_deposits_amount,total_equity_amount,total_UT,customer_revenue,age,avg_balance,num_accs,balance_usd,available_balance_usd
148,4,1,15,1,1,67,3,1229.33,709.64,0.88,28332,28332,2.9202690498E8,10,9,231697.88,1979774.37,9048360.0,8079228.0,22,1002.01,4,581.33,2247.8
463,4,1,17,1,0,28,4,10159.61,1202.23,3.58,97428,0,2.1270324083E8,4,6,8457003.0,8405245.0,8589421.0,7210788.0,54,1181.865,2,2154.23,838.17
471,0,0,5,1,0,52,2,52.0,1467.79,2.35,106068,1060,4.4269459686E8,8,9,1606463.0,5484075.09,3431393.0,7249848.02,36,5238.02,1,5238.02,1924.28
496,0,0,1,18,0,108,5,5081.31,49.16,0.04,69360,69360,8.6094333863E8,5,4,9366416.0,7425938.0,6857042.78,4044357.0,61,1424.46,1,1424.46,919.08
833,4,0,17,33,1,147,1,4171.9,946.31,0.54,95544,955,6.5867073154E8,10,8,4417594.81,1782286.12,3998838.0,7631038.0,65,415.0,1,415.0,895.37
1088,2,2,53,49,0,52,4,11780.41,199.02,0.32,75048,0,9.3016475685E8,8,3,2275684.32,1252601.0,498446.29,7084018.0,65,1332.28,1,1332.28,1262.51
1238,4,0,49,57,1,68,1,198.0,1363.77,1.67,46596,0,1.1610549949E8,2,7,4674719.93,6838865.0,3268432.51,4055899.25,24,261.22,1,261.22,250.14
1342,0,2,73,24,0,88,3,14296.34,135.68,0.13,115620,0,5.4065295252E8,10,9,1179828.14,8551756.0,2715156.85,8378624.0,38,2885.285,2,5653.07,470.46
1580,0,0,85,38,1,73,1,8021.92,1608.98,1.84,83856,83856,1.2967405265E8,2,3,9909585.0,1669979.47,2867585.13,8552203.0,36,4850.06,1,4850.06,922.52
1591,0,0,6,38,1,82,3,8275.41,521.87,0.53,87168,0,1.3138714558E8,3,8,5157854.14,6313649.0,3477629.02,1294329.94,63,1224.18,1,1224.18,698990.0


In [0]:
telco_gold_features = (spark.table("telco_gold")
                            .select('cust_id', 'is_pre_paid', 'number_payment_delays_last12mo', 'pct_increase_annual_number_of_delays_last_3_year', 'phone_bill_amt', \
                                    'avg_phone_bill_amt_lst12mo')).dropDuplicates(['cust_id'])
display(telco_gold_features)

cust_id,is_pre_paid,number_payment_delays_last12mo,pct_increase_annual_number_of_delays_last_3_year,phone_bill_amt,avg_phone_bill_amt_lst12mo
97004,0,0,0,19.99,19.99
99168,0,0,0,19.99,19.99
97186,0,0,5,19.99,19.99
12046,0,0,0,49.99,49.99
18800,0,1,9,19.99,19.99
49308,0,1,0,34.99,34.99
38220,0,0,0,89.99,89.99
30970,0,0,0,34.99,34.99
31035,0,3,0,89.99,89.99
24347,0,0,8,49.99,49.99


In [0]:
fund_trans_gold_features = spark.table("fund_trans_gold").dropDuplicates(['cust_id'])

for c in ['12m', '6m', '3m']:
  fund_trans_gold_features = fund_trans_gold_features.withColumn('tot_txn_cnt_'+c, col('sent_txn_cnt_'+c)+col('rcvd_txn_cnt_'+c))\
                                                     .withColumn('tot_txn_amt_'+c, col('sent_txn_amt_'+c)+col('rcvd_txn_amt_'+c))

fund_trans_gold_features = fund_trans_gold_features.withColumn('ratio_txn_amt_3m_12m', F.when(col('tot_txn_amt_12m')==0, 0).otherwise(col('tot_txn_amt_3m')/col('tot_txn_amt_12m')))\
                                                   .withColumn('ratio_txn_amt_6m_12m', F.when(col('tot_txn_amt_12m')==0, 0).otherwise(col('tot_txn_amt_6m')/col('tot_txn_amt_12m')))\
                                                   .na.fill(0)
display(fund_trans_gold_features)

cust_id,dist_payer_cnt_12m,sent_txn_cnt_12m,sent_txn_amt_12m,sent_amt_avg_12m,dist_payee_cnt_12m,rcvd_txn_cnt_12m,rcvd_txn_amt_12m,rcvd_amt_avg_12m,dist_payer_cnt_6m,sent_txn_cnt_6m,sent_txn_amt_6m,sent_amt_avg_6m,dist_payee_cnt_6m,rcvd_txn_cnt_6m,rcvd_txn_amt_6m,rcvd_amt_avg_6m,dist_payer_cnt_3m,sent_txn_cnt_3m,sent_txn_amt_3m,sent_amt_avg_3m,dist_payee_cnt_3m,rcvd_txn_cnt_3m,rcvd_txn_amt_3m,rcvd_amt_avg_3m,tot_txn_cnt_12m,tot_txn_amt_12m,tot_txn_cnt_6m,tot_txn_amt_6m,tot_txn_cnt_3m,tot_txn_amt_3m,ratio_txn_amt_3m_12m,ratio_txn_amt_6m_12m
148,0,0,0.0,0.0,1,1,168.74,168.74,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
463,0,0,0.0,0.0,1,2,388.11,194.055,0,0,0.0,0.0,1,1,166.35,166.35,0,0,0.0,0.0,1,1,166.35,166.35,0,0.0,0,0.0,0,0.0,0.0,0.0
471,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
496,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
833,0,0,0.0,0.0,1,20,4051.290000000001,202.56450000000004,0,0,0.0,0.0,1,11,2191.55,199.2318181818182,0,0,0.0,0.0,1,3,618.3,206.1,0,0.0,0,0.0,0,0.0,0.0,0.0
1088,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
1238,0,0,0.0,0.0,1,5,805.66,161.132,0,0,0.0,0.0,1,3,484.08,161.35999999999999,0,0,0.0,0.0,1,2,305.15999999999997,152.57999999999998,0,0.0,0,0.0,0,0.0,0.0,0.0
1342,0,0,0.0,0.0,1,2,431.7,215.85,0,0,0.0,0.0,1,1,197.7,197.7,0,0,0.0,0.0,1,1,197.7,197.7,0,0.0,0,0.0,0,0.0,0.0,0.0
1580,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
1591,0,0,0.0,0.0,1,10,2026.6600000000003,202.66600000000003,0,0,0.0,0.0,1,5,1004.0199999999999,200.80399999999997,0,0,0.0,0.0,1,3,577.2099999999999,192.4033333333333,0,0.0,0,0.0,0,0.0,0.0,0.0


In [0]:
feature_df = customer_gold_features.join(telco_gold_features.alias('telco'), "cust_id", how="left")
feature_df = feature_df.join(fund_trans_gold_features, "cust_id", how="left")
display(feature_df)

cust_id,education,marital_status,months_current_address,months_employment,is_resident,tenure_months,product_cnt,tot_rel_bal,revenue_tot,revenue_12m,income_annual,tot_assets,overdraft_balance_amount,overdraft_number,total_deposits_number,total_deposits_amount,total_equity_amount,total_UT,customer_revenue,age,avg_balance,num_accs,balance_usd,available_balance_usd,is_pre_paid,number_payment_delays_last12mo,pct_increase_annual_number_of_delays_last_3_year,phone_bill_amt,avg_phone_bill_amt_lst12mo,dist_payer_cnt_12m,sent_txn_cnt_12m,sent_txn_amt_12m,sent_amt_avg_12m,dist_payee_cnt_12m,rcvd_txn_cnt_12m,rcvd_txn_amt_12m,rcvd_amt_avg_12m,dist_payer_cnt_6m,sent_txn_cnt_6m,sent_txn_amt_6m,sent_amt_avg_6m,dist_payee_cnt_6m,rcvd_txn_cnt_6m,rcvd_txn_amt_6m,rcvd_amt_avg_6m,dist_payer_cnt_3m,sent_txn_cnt_3m,sent_txn_amt_3m,sent_amt_avg_3m,dist_payee_cnt_3m,rcvd_txn_cnt_3m,rcvd_txn_amt_3m,rcvd_amt_avg_3m,tot_txn_cnt_12m,tot_txn_amt_12m,tot_txn_cnt_6m,tot_txn_amt_6m,tot_txn_cnt_3m,tot_txn_amt_3m,ratio_txn_amt_3m_12m,ratio_txn_amt_6m_12m
148,4,1,15,1,1,67,3,1229.33,709.64,0.88,28332,28332,2.9202690498E8,10,9,231697.88,1979774.37,9048360.0,8079228.0,22,1002.01,4,581.33,2247.8,0,0,0,49.99,49.99,0,0,0.0,0.0,1,1,168.74,168.74,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
463,4,1,17,1,0,28,4,10159.61,1202.23,3.58,97428,0,2.1270324083E8,4,6,8457003.0,8405245.0,8589421.0,7210788.0,54,1181.865,2,2154.23,838.17,0,1,0,49.99,49.99,0,0,0.0,0.0,1,2,388.11,194.055,0,0,0.0,0.0,1,1,166.35,166.35,0,0,0.0,0.0,1,1,166.35,166.35,0,0.0,0,0.0,0,0.0,0.0,0.0
471,0,0,5,1,0,52,2,52.0,1467.79,2.35,106068,1060,4.4269459686E8,8,9,1606463.0,5484075.09,3431393.0,7249848.02,36,5238.02,1,5238.02,1924.28,0,0,0,49.99,49.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
496,0,0,1,18,0,108,5,5081.31,49.16,0.04,69360,69360,8.6094333863E8,5,4,9366416.0,7425938.0,6857042.78,4044357.0,61,1424.46,1,1424.46,919.08,0,0,0,19.99,19.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
833,4,0,17,33,1,147,1,4171.9,946.31,0.54,95544,955,6.5867073154E8,10,8,4417594.81,1782286.12,3998838.0,7631038.0,65,415.0,1,415.0,895.37,0,4,0,49.99,49.99,0,0,0.0,0.0,1,20,4051.290000000001,202.56450000000004,0,0,0.0,0.0,1,11,2191.55,199.2318181818182,0,0,0.0,0.0,1,3,618.3,206.1,0,0.0,0,0.0,0,0.0,0.0,0.0
1088,2,2,53,49,0,52,4,11780.41,199.02,0.32,75048,0,9.3016475685E8,8,3,2275684.32,1252601.0,498446.29,7084018.0,65,1332.28,1,1332.28,1262.51,null,null,null,null,null,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
1238,4,0,49,57,1,68,1,198.0,1363.77,1.67,46596,0,1.1610549949E8,2,7,4674719.93,6838865.0,3268432.51,4055899.25,24,261.22,1,261.22,250.14,1,0,0,26.98,26.98,0,0,0.0,0.0,1,5,805.66,161.132,0,0,0.0,0.0,1,3,484.08,161.35999999999999,0,0,0.0,0.0,1,2,305.15999999999997,152.57999999999998,0,0.0,0,0.0,0,0.0,0.0,0.0
1342,0,2,73,24,0,88,3,14296.34,135.68,0.13,115620,0,5.4065295252E8,10,9,1179828.14,8551756.0,2715156.85,8378624.0,38,2885.285,2,5653.07,470.46,0,0,0,19.99,19.99,0,0,0.0,0.0,1,2,431.7,215.85,0,0,0.0,0.0,1,1,197.7,197.7,0,0,0.0,0.0,1,1,197.7,197.7,0,0.0,0,0.0,0,0.0,0.0,0.0
1580,0,0,85,38,1,73,1,8021.92,1608.98,1.84,83856,83856,1.2967405265E8,2,3,9909585.0,1669979.47,2867585.13,8552203.0,36,4850.06,1,4850.06,922.52,0,4,0,124.99,124.99,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0,0.0,0.0,0,0.0,0,0.0,0,0.0,0.0,0.0
1591,0,0,6,38,1,82,3,8275.41,521.87,0.53,87168,0,1.3138714558E8,3,8,5157854.14,6313649.0,3477629.02,1294329.94,63,1224.18,1,1224.18,698990.0,0,0,12,34.99,34.99,0,0,0.0,0.0,1,10,2026.6600000000003,202.66600000000003,0,0,0.0,0.0,1,5,1004.0199999999999,200.80399999999997,0,0,0.0,0.0,1,3,577.2099999999999,192.4033333333333,0,0.0,0,0.0,0,0.0,0.0,0.0



# Databricks Feature Store

<img src="https://github.com/QuentinAmbard/databricks-demo/raw/main/product_demos/mlops-end2end-flow-feature-store.png" style="float:right" width="650" />

Once our features are ready, we'll save them in Databricks Feature Store. 

Under the hood, feature store are backed by a Delta Lake table. This will allow discoverability and reusability of our feature across our organization, increasing team efficiency.


Databricks Feature Store brings advanced capabilities to accelerate and simplify your ML journey, such as point in time support and online-store, fetching your features within ms for real time Serving. 

### Why use Databricks Feature Store?

Databricks Feature Store is fully integrated with other components of Databricks.

* **Discoverability**. The Feature Store UI, accessible from the Databricks workspace, lets you browse and search for existing features.

* **Lineage**. When you create a feature table with Feature Store, the data sources used to create the feature table are saved and accessible. For each feature in a feature table, you can also access the models, notebooks, jobs, and endpoints that use the feature.

* **Batch and Online feature lookup for real time serving**. When you use features from Feature Store to train a model, the model is packaged with feature metadata. When you use the model for batch scoring or online inference, it automatically retrieves features from Feature Store. The caller does not need to know about them or include logic to look up or join features to score new data. This makes model deployment and updates much easier.

* **Point-in-time lookups**. Feature Store supports time series and event-based use cases that require point-in-time correctness.


For more details about Databricks Feature Store, run `dbdemos.install('feature-store')`

In [0]:
from databricks import feature_store
fs = feature_store.FeatureStoreClient()

# Drop the fs table if it was already existing to cleanup the demo state
drop_fs_table(f"{catalog}.{db}.credit_decisioning_features")
  
fs.create_table(
    name=f"{catalog}.{db}.credit_decisioning_features",
    primary_keys=["cust_id"],
    df=feature_df,
    description="Features for Credit Decisioning.")

2025/12/04 19:45:33 WARNING databricks.ml_features._compute_client._compute_client: Deleting a feature table can lead to unexpected failures in upstream producers and downstream consumers (models, endpoints, and scheduled jobs).
2025/12/04 19:45:41 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['cust_id'] of table 'main.dbdemos_fsi_credit_decisioning.credit_decisioning_features' to NOT NULL.
2025/12/04 19:45:42 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['cust_id'] on table 'main.dbdemos_fsi_credit_decisioning.credit_decisioning_features'.
2025/12/04 19:45:50 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'main.dbdemos_fsi_credit_decisioning.credit_decisioning_features'.


<FeatureTable: name='main.dbdemos_fsi_credit_decisioning.credit_decisioning_features', table_id='cf01e9e7-d017-4787-86e1-592ce1ee46bd', description='Features for Credit Decisioning.', primary_keys=['cust_id'], partition_columns=[], features=['cust_id',
 'education',
 'marital_status',
 'months_current_address',
 'months_employment',
 'is_resident',
 'tenure_months',
 'product_cnt',
 'tot_rel_bal',
 'revenue_tot',
 'revenue_12m',
 'income_annual',
 'tot_assets',
 'overdraft_balance_amount',
 'overdraft_number',
 'total_deposits_number',
 'total_deposits_amount',
 'total_equity_amount',
 'total_UT',
 'customer_revenue',
 'age',
 'avg_balance',
 'num_accs',
 'balance_usd',
 'available_balance_usd',
 'is_pre_paid',
 'number_payment_delays_last12mo',
 'pct_increase_annual_number_of_delays_last_3_year',
 'phone_bill_amt',
 'avg_phone_bill_amt_lst12mo',
 'dist_payer_cnt_12m',
 'sent_txn_cnt_12m',
 'sent_txn_amt_12m',
 'sent_amt_avg_12m',
 'dist_payee_cnt_12m',
 'rcvd_txn_cnt_12m',
 'rcvd_txn_


## Next steps

After creating our features and storing them in the Databricks Feature Store, we can now proceed to the [03.2-AutoML-credit-decisioning]($./03.2-AutoML-credit-decisioning) and build out credit decisioning model.